In [1]:
# ================================================================
# VERSION 11: THE DEFINITIVE AI IMAGE DETECTOR
# ================================================================
# What works (proven): DINOv2-BASE@518 (0.905) + SigLIP (0.90) + CLIP (0.89)
# What failed: DINOv2-Large, DIRE, Label Smoothing, CutMix, Stage 3
# Strategy: 3 strong models -> Isotonic Calibration -> Hill Climbing
# ================================================================
import subprocess, sys
for p in [
    'numpy pandas opencv-python-headless Pillow tqdm scipy PyWavelets '
    'matplotlib seaborn scikit-learn xgboost joblib',
    'torch torchvision --index-url https://download.pytorch.org/whl/cu118',
    'git+https://github.com/openai/CLIP.git',
    'timm albumentations transformers safetensors',
    '--upgrade typing_extensions',
]:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + p.split(), check=False)
print('All packages ready.')

All packages ready.


In [2]:
!pip install sentencepiece protobuf huggingface_hub

In [1]:
# ================================================================
# CELL 2: IMPORTS, CONFIG, GPU (FIXED & CRASH-PROOFED)
# ================================================================
import os

# 🚨 PREVENT HUGGINGFACE RATE LIMIT CRASHES & ENSURE DETERMINISM
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys, io, warnings, random, gc, copy, time, json
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
from scipy.fft import fft2, fftshift
from scipy.stats import rankdata
from scipy.special import expit
from sklearn.isotonic import IsotonicRegression
import pywt

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, confusion_matrix, roc_auc_score, roc_curve,
                             precision_score, recall_score, accuracy_score)
from sklearn.feature_selection import VarianceThreshold
import xgboost as xgb
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import clip
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 🚨 ADDED MISSING IMPORT FOR CELL 10
from transformers import SiglipModel

warnings.filterwarnings('ignore')

def gpu_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def print_vram():
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated()/1e9
        free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9
        print(f'  VRAM: {used:.1f}GB used | {free:.1f}GB free')

def ensure_vram(min_gb=4.0):
    gpu_cleanup()
    print_vram()

# Path detection
def detect_paths():
    for csv in Path('..').rglob('train.csv'):
        test = csv.parent / 'test.csv'
        if test.exists():
            img = csv.parent / 'images_final_sample'
            if not img.exists():
                for d in csv.parent.rglob('images_final_sample'):
                    img = d; break
            if not img.exists(): img = csv.parent
            return img, csv, test
    raise FileNotFoundError('No train.csv found')

IMAGE_DIR, TRAIN_CSV, TEST_CSV = detect_paths()

SEED = 42
CACHE_DIR = Path('./cache_v11')
MODEL_DIR = Path('./models_v11')
CHKPT_DIR = Path('./chkpt_v11')
for d in [CACHE_DIR, MODEL_DIR, CHKPT_DIR]: d.mkdir(exist_ok=True)

N_FOLDS = 5
N_TTA = 5
FORCE_FRESH = False

SIGLIP_DIM = 1152
CLIP_DIM = 768
DINO_DIM = 768  # BASE not Large
CNN_DIM = 1280
FORENSIC_DIM = 114

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]
DINO_MEAN = [0.485, 0.456, 0.406]
DINO_STD = [0.229, 0.224, 0.225]
SIGLIP_MEAN = [0.5, 0.5, 0.5]
SIGLIP_STD = [0.5, 0.5, 0.5]

def set_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seeds()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Strategy: DINOv2-BASE@518 + SigLIP@224 + CLIP@224')
print(f'Ensemble: Isotonic Calibration -> Hill Climbing')
print('CELL 2 COMPLETE')

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 3090
VRAM: 25.4 GB
Strategy: DINOv2-BASE@518 + SigLIP@224 + CLIP@224
Ensemble: Isotonic Calibration -> Hill Climbing
CELL 2 COMPLETE


In [2]:
# ================================================================
# CELL 3: DATASET LOADING
# ================================================================
set_seeds()
df_train = pd.read_csv(TRAIN_CSV)
df_test = pd.read_csv(TEST_CSV)

sample = df_train['image_id'].iloc[0]
img_dir_found = None
for r, _, files in os.walk('..'):
    if sample in files:
        img_dir_found = Path(r); break
if img_dir_found: IMAGE_DIR = img_dir_found
print(f'Images: {IMAGE_DIR}')

df_train['filepath'] = df_train['image_id'].apply(lambda x: str(IMAGE_DIR / x))
df_test['filepath'] = df_test['image_id'].apply(lambda x: str(IMAGE_DIR / x))

found = sum(Path(p).exists() for p in df_train['filepath'][:50])
assert found >= 45, f'Images missing! Found {found}/50'

train_paths = df_train['filepath'].tolist()
test_paths = df_test['filepath'].tolist()
y_all = df_train['ground_truth'].values.astype(np.float32)

print(f'Train: {len(train_paths)} | Test: {len(test_paths)}')
print(f'Class: {y_all.mean()*100:.1f}% AI | {(1-y_all.mean())*100:.1f}% Real')
print('CELL 3 COMPLETE')

Images: ../data/DCU 2026 ML challenge - external 2/images/images_final_sample
Train: 4800 | Test: 2058
Class: 48.2% AI | 51.8% Real
CELL 3 COMPLETE


In [20]:
# ================================================================
# CELL 4: IMAGE LOADING + AUGMENTATIONS + MIXUP (FINAL API FIX)
# No CutMix (over-regularized in v10), no label smoothing
# ================================================================
def load_image_pil(path):
    try: return Image.open(str(path)).convert('RGB')
    except Exception:
        try:
            img = cv2.imread(str(path))
            if img is not None: return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        except: pass
    return None

def _gauss_noise():
    try: return A.GaussNoise(std_range=(0.01, 0.05), p=0.2)
    except: return A.GaussNoise(var_limit=(5.0, 30.0), p=0.2)

def _coarse_dropout():
    try: return A.CoarseDropout(num_holes_range=(1,4), hole_height_range=(8,32), hole_width_range=(8,32), p=0.2)
    except: return A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2)

def _img_compression():
    try: return A.ImageCompression(quality_range=(65,100), p=0.3)
    except: return A.ImageCompression(quality_lower=65, quality_upper=100, p=0.3)

def make_train_tfm(mean, std, size=224):
    return A.Compose([
        # RandomResizedCrop strictly wants 'size'
        A.RandomResizedCrop(size=(size, size), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
        _gauss_noise(),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        _img_compression(),
        A.Normalize(mean=mean, std=std),
        _coarse_dropout(),
        ToTensorV2(),
    ])

def make_val_tfm(mean, std, size=224):
    return A.Compose([
        # 🚨 FIXED: Resize strictly wants 'height' and 'width'
        A.Resize(height=size, width=size),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])

def make_tta_tfm(mean, std, size=224):
    return A.Compose([
        # RandomResizedCrop strictly wants 'size'
        A.RandomResizedCrop(size=(size, size), scale=(0.9, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])

class AlbuDataset(Dataset):
    def __init__(self, paths, labels, transform, img_size=224):
        self.paths=paths; self.labels=labels; self.transform=transform; self.img_size=img_size
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = load_image_pil(self.paths[idx])
        if img is None: img = Image.new('RGB', (self.img_size, self.img_size), (128,128,128))
        arr = np.array(img)
        t = self.transform(image=arr)['image']
        if not isinstance(t, torch.Tensor): t = torch.from_numpy(t.transpose(2,0,1)).float()
        lbl = float(self.labels[idx]) if self.labels is not None else 0.0
        return t, torch.tensor(lbl, dtype=torch.float32)

def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

print('Augmentations ready (train/val/tta + Mixup).')
print('CELL 4 COMPLETE')

Augmentations ready (train/val/tta + Mixup).
CELL 4 COMPLETE


In [21]:
# ================================================================
# CELL 5: SigLIP So400m FEATURE EXTRACTION (1152-dim) (FIXED)
# ================================================================
from transformers import SiglipModel, SiglipProcessor
gpu_cleanup(); ensure_vram(4.0); set_seeds()

def extract_siglip_features(image_paths, batch_size=32):
    m = SiglipModel.from_pretrained('google/siglip-so400m-patch14-224')
    proc = SiglipProcessor.from_pretrained('google/siglip-so400m-patch14-224')
    vm = m.vision_model.to(DEVICE).eval()
    all_f = []
    
    for i in tqdm(range(0, len(image_paths), batch_size), desc='SigLIP'):
        batch = [load_image_pil(p) or Image.new('RGB',(224,224),(128,128,128))
                 for p in image_paths[i:i+batch_size]]
        inp = proc(images=batch, return_tensors='pt', padding=True)
        
        with torch.no_grad(), autocast():
            # 🚨 FIXED: Matching Cell 10 logic. Using last_hidden_state instead of pooler_output
            out = vm(pixel_values=inp['pixel_values'].to(DEVICE)).last_hidden_state
            f = out.mean(dim=1).float()
            
        all_f.append(F.normalize(f, dim=-1).cpu().numpy())
        
    del m, vm, proc; gpu_cleanup()
    return np.vstack(all_f).astype(np.float32)

if FORCE_FRESH or not (CACHE_DIR/'siglip_train.npy').exists():
    print('Extracting SigLIP...')
    siglip_train = extract_siglip_features(train_paths)
    siglip_test = extract_siglip_features(test_paths)
    np.save(CACHE_DIR/'siglip_train.npy', siglip_train)
    np.save(CACHE_DIR/'siglip_test.npy', siglip_test)
else:
    siglip_train = np.load(CACHE_DIR/'siglip_train.npy')
    siglip_test = np.load(CACHE_DIR/'siglip_test.npy')

print(f'SigLIP: {siglip_train.shape}'); print('CELL 5 COMPLETE')

  VRAM: 0.0GB used | 25.4GB free
SigLIP: (4800, 1152)
CELL 5 COMPLETE


In [22]:
# ================================================================
# CELL 6: CLIP ViT-L/14 FEATURE EXTRACTION (768-dim)
# ================================================================
gpu_cleanup(); ensure_vram(4.0); set_seeds()

def extract_clip_features(image_paths, batch_size=64):
    m, prep = clip.load('ViT-L/14', device='cpu')
    m = m.float().to(DEVICE).eval()
    all_f = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='CLIP'):
        batch = [prep(load_image_pil(p) or Image.new('RGB',(224,224)))
                 for p in image_paths[i:i+batch_size]]
        with torch.no_grad():
            f = m.encode_image(torch.stack(batch).to(DEVICE)).float()
        all_f.append(F.normalize(f, dim=-1).cpu().numpy())
    del m; gpu_cleanup()
    return np.vstack(all_f).astype(np.float32)

if FORCE_FRESH or not (CACHE_DIR/'clip_train.npy').exists():
    print('Extracting CLIP...')
    clip_train = extract_clip_features(train_paths)
    clip_test = extract_clip_features(test_paths)
    np.save(CACHE_DIR/'clip_train.npy', clip_train)
    np.save(CACHE_DIR/'clip_test.npy', clip_test)
else:
    clip_train = np.load(CACHE_DIR/'clip_train.npy')
    clip_test = np.load(CACHE_DIR/'clip_test.npy')
print(f'CLIP: {clip_train.shape}'); print('CELL 6 COMPLETE')

  VRAM: 0.0GB used | 25.4GB free
CLIP: (4800, 768)
CELL 6 COMPLETE


In [23]:
# ================================================================
# CELL 7: DINOv2-BASE @ 518px (768-dim) — V5's BEST model (0.9048)
# NOT Large! Base proved superior on 4800 samples.
# ================================================================
gpu_cleanup(); ensure_vram(4.0); set_seeds()

def extract_dino_features(image_paths, batch_size=16):
    m = timm.create_model('vit_base_patch14_dinov2', pretrained=True, num_classes=0)
    m = m.to(DEVICE).eval()
    tfm = T.Compose([T.Resize(518, interpolation=T.InterpolationMode.BICUBIC),
                      T.CenterCrop(518), T.ToTensor(), T.Normalize(DINO_MEAN, DINO_STD)])
    all_f = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='DINOv2-B'):
        batch = [tfm(load_image_pil(p) or Image.new('RGB',(518,518)))
                 for p in image_paths[i:i+batch_size]]
        with torch.no_grad():
            with autocast(): f = m(torch.stack(batch).to(DEVICE))
        all_f.append(F.normalize(f.float(), dim=-1).cpu().numpy())
    del m; gpu_cleanup()
    return np.vstack(all_f).astype(np.float32)

if FORCE_FRESH or not (CACHE_DIR/'dino_train.npy').exists():
    print('Extracting DINOv2-Base@518...')
    dino_train = extract_dino_features(train_paths)
    dino_test = extract_dino_features(test_paths)
    np.save(CACHE_DIR/'dino_train.npy', dino_train)
    np.save(CACHE_DIR/'dino_test.npy', dino_test)
else:
    dino_train = np.load(CACHE_DIR/'dino_train.npy')
    dino_test = np.load(CACHE_DIR/'dino_test.npy')
print(f'DINOv2-B: {dino_train.shape}'); print('CELL 7 COMPLETE')

  VRAM: 0.0GB used | 25.4GB free
DINOv2-B: (4800, 768)
CELL 7 COMPLETE


In [24]:
# ================================================================
# CELL 8: CNN (1280-dim) + FORENSIC (114-dim)
# ================================================================
from joblib import Parallel, delayed
from scipy.ndimage import median_filter
gpu_cleanup(); ensure_vram(2.0); set_seeds()

def extract_cnn_features(image_paths, batch_size=128):
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier = nn.Identity()
    m = m.to(DEVICE).eval()
    tfm = T.Compose([T.Resize((224,224)), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    all_f = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='CNN'):
        batch = [tfm(load_image_pil(p) or Image.new('RGB',(224,224)))
                 for p in image_paths[i:i+batch_size]]
        with torch.no_grad(): f = m(torch.stack(batch).to(DEVICE))
        all_f.append(F.normalize(f.float(), dim=-1).cpu().numpy())
    del m; gpu_cleanup()
    return np.vstack(all_f).astype(np.float32)

def _ela(img_np):
    feats=[]; pil=Image.fromarray(img_np.astype(np.uint8))
    for q in [90,75,50]:
        buf=io.BytesIO(); pil.save(buf,'JPEG',quality=q); buf.seek(0)
        ela=np.abs(img_np.astype(np.float64)-np.array(Image.open(buf),dtype=np.float64))
        for c in range(3): ch=ela[:,:,c]; feats.extend([ch.mean(),ch.std()])
        g=ela.mean(2); feats.extend([np.percentile(g,95),np.percentile(g,5)])
    return np.array(feats,dtype=np.float32)

def _fft(img_np):
    mag=np.log1p(np.abs(fftshift(fft2(img_np.mean(2)))))
    h,w=mag.shape; cy,cx=h//2,w//2; mr=min(cy,cx)
    y,x=np.indices((h,w)); r=np.sqrt((x-cx)**2+(y-cy)**2)
    bins=np.array([mag[(r>=i*mr/30)&(r<(i+1)*mr/30)].mean()
                   if ((r>=i*mr/30)&(r<(i+1)*mr/30)).any() else 0.0 for i in range(30)])
    yg,xg=np.ogrid[-cy:h-cy,-cx:w-cx]; rsq=yg**2+xg**2
    return np.array(list(bins)+[mag.mean(),mag.std(),
        mag[rsq<(mr*0.2)**2].sum(),
        mag[(rsq>=(mr*0.2)**2)&(rsq<(mr*0.5)**2)].sum()],dtype=np.float32)

def _noise(img_np):
    gray=img_np.mean(2); feats=[]
    for w in ['db1','db2']:
        _,(cH,cV,cD)=pywt.dwt2(gray,w)
        for d in [cH,cV,cD]: feats.extend([np.abs(d).mean(),d.std(),np.percentile(np.abs(d),99),np.mean(d**2)])
    dn=median_filter(gray,size=3); n=gray-dn
    feats.extend([n.mean(),n.std(),np.mean(n**2),np.percentile(n,1),np.percentile(n,99)])
    return np.array(feats[:40],dtype=np.float32)

def get_forensic(path):
    img = load_image_pil(path)
    if img is None: return np.zeros(FORENSIC_DIM,dtype=np.float32)
    img = img.resize((256,256), Image.Resampling.LANCZOS)
    arr = np.array(img, dtype=np.float64)
    f=np.concatenate([_ela(arr),_fft(arr),_noise(arr)])
    if len(f)<FORENSIC_DIM: f=np.pad(f,(0,FORENSIC_DIM-len(f)))
    return f[:FORENSIC_DIM].astype(np.float32)

if FORCE_FRESH or not (CACHE_DIR/'cnn_train.npy').exists():
    cnn_train=extract_cnn_features(train_paths); cnn_test=extract_cnn_features(test_paths)
    np.save(CACHE_DIR/'cnn_train.npy',cnn_train); np.save(CACHE_DIR/'cnn_test.npy',cnn_test)
else:
    cnn_train=np.load(CACHE_DIR/'cnn_train.npy'); cnn_test=np.load(CACHE_DIR/'cnn_test.npy')

if FORCE_FRESH or not (CACHE_DIR/'forensic_train.npy').exists():
    print('Extracting Forensic (parallel)...')
    forensic_train=np.array(Parallel(n_jobs=-1)(delayed(get_forensic)(p) for p in tqdm(train_paths,desc='F-Tr')))
    forensic_test=np.array(Parallel(n_jobs=-1)(delayed(get_forensic)(p) for p in tqdm(test_paths,desc='F-Te')))
    np.save(CACHE_DIR/'forensic_train.npy',forensic_train); np.save(CACHE_DIR/'forensic_test.npy',forensic_test)
else:
    forensic_train=np.load(CACHE_DIR/'forensic_train.npy'); forensic_test=np.load(CACHE_DIR/'forensic_test.npy')
print(f'CNN: {cnn_train.shape} | Forensic: {forensic_train.shape}')
print('CELL 8 COMPLETE')

  VRAM: 0.0GB used | 25.4GB free
CNN: (4800, 1280) | Forensic: (4800, 114)
CELL 8 COMPLETE


In [25]:
# ================================================================
# CELL 9: TRAINING INFRASTRUCTURE — FOCAL + MIXUP + ONECYCLE (FIXED)
# ================================================================
results_tracker = {}
oof_store = {}
test_pred_store = {}
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

class FocalBCEWithLogitsLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        # Pure BCE for true probability
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        # Standard Focal scaling
        focal_loss = ((1 - pt) ** self.gamma) * bce
        # Apply pos_weight manually to avoid warping
        if self.pos_weight is not None:
            weight = targets * self.pos_weight + (1 - targets)
            focal_loss = focal_loss * weight
        return focal_loss.mean()

def run_epoch_train(model, loader, optimizer, scaler, criterion, scheduler=None):
    model.train(); tot=0.0
    for imgs, labels in loader:
        imgs=imgs.to(DEVICE); labels=labels.to(DEVICE).float()
        r=random.random()
        
        if r < 0.5:
            # 🚨 FIXED: Blend the soft labels (y_mix), not the non-linear Focal losses!
            imgs, ya, yb, lam = mixup_data(imgs, labels)
            y_mix = lam * ya + (1 - lam) * yb
            with autocast(): loss = criterion(model(imgs), y_mix)
        else:
            with autocast(): loss = criterion(model(imgs), labels)
            
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        if scheduler is not None: scheduler.step()
        tot+=loss.item()*len(labels)
    return tot/len(loader.dataset)

def eval_epoch(model, loader):
    model.eval(); preds=[]; trues=[]
    with torch.no_grad():
        for imgs,labels in loader:
            imgs=imgs.to(DEVICE)
            with autocast(): logits=model(imgs)
            p=torch.sigmoid(logits).float().cpu().numpy()
            preds.extend(np.clip(p,0,1).tolist())
            trues.extend(labels.numpy().tolist())
    p_arr=np.array(preds)
    return f1_score(trues,(p_arr>=0.5).astype(int),zero_division=0), p_arr

def train_fold(tag, fold, model, tr_ds, va_ds, criterion,
               freeze_fn, unfreeze_fn, get_opt_fn, batch_size=16):
    nw = min(8, os.cpu_count() or 4)
    tr_ld=DataLoader(tr_ds,batch_size=batch_size,shuffle=True,num_workers=nw,pin_memory=True,drop_last=True)
    va_ld=DataLoader(va_ds,batch_size=batch_size,shuffle=False,num_workers=nw,pin_memory=True)
    scaler=GradScaler(enabled=True)
    best_f1=0; best_vp=None; patience=5; no_imp=0
    pt_path = MODEL_DIR/f'{tag}_fold_{fold}.pt'

    # Stage 1: Head only (10 epochs)
    freeze_fn(model)
    opt1=torch.optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=1e-4,weight_decay=0.01)
    sch1=torch.optim.lr_scheduler.OneCycleLR(opt1,max_lr=1e-3,epochs=10,steps_per_epoch=len(tr_ld))
    for ep in range(1,11):
        run_epoch_train(model,tr_ld,opt1,scaler,criterion,sch1)
        vf,vp=eval_epoch(model,va_ld)
        if vf>best_f1:
            best_f1=vf; best_vp=vp.copy(); no_imp=0
            torch.save({k:v.cpu().clone() for k,v in model.state_dict().items()},pt_path)
        else: no_imp+=1
        if no_imp>=patience: break
    s1=best_f1; no_imp=0

    # Stage 2: Unfreeze last blocks (15 epochs)
    state=torch.load(pt_path,map_location='cpu'); model.load_state_dict(state); del state
    unfreeze_fn(model)
    opt2=get_opt_fn(model)
    # 🚨 FIXED: Extracting max_lrs as a list so PyTorch respects parameter groups
    max_lrs = [group['lr'] for group in opt2.param_groups] 
    sch2=torch.optim.lr_scheduler.OneCycleLR(opt2,max_lr=max_lrs,epochs=15,steps_per_epoch=len(tr_ld))
    for ep in range(1,16):
        run_epoch_train(model,tr_ld,opt2,scaler,criterion,sch2)
        vf,vp=eval_epoch(model,va_ld)
        if vf>best_f1:
            best_f1=vf; best_vp=vp.copy(); no_imp=0
            torch.save({k:v.cpu().clone() for k,v in model.state_dict().items()},pt_path)
        else: no_imp+=1
        if no_imp>=patience: break

    print(f'   Fold {fold+1}: S1={s1:.4f} -> S2={best_f1:.4f}')
    return best_f1, best_vp

def run_tta_disk(tag, model_fn, tta_tfm, img_size=224, batch_size=16):
    test_proba=np.zeros(len(test_paths)); folds_done=0
    nw=min(8, os.cpu_count() or 4)
    for fold in range(N_FOLDS):
        pt=MODEL_DIR/f'{tag}_fold_{fold}.pt'
        if not pt.exists(): print(f'  SKIP {pt}'); continue
        print(f'  TTA Fold {fold+1}...')
        state=torch.load(pt,map_location='cpu')
        model=model_fn(); model.load_state_dict(state); del state
        model=model.to(DEVICE).eval()
        ds=AlbuDataset(test_paths,[0.0]*len(test_paths),tta_tfm,img_size=img_size)
        ld=DataLoader(ds,batch_size=batch_size,shuffle=False,num_workers=nw)
        for _ in range(N_TTA):
            preds=[]
            with torch.no_grad():
                for imgs,_ in ld:
                    with autocast(): logits=model(imgs.to(DEVICE))
                    preds.extend(torch.sigmoid(logits).float().cpu().numpy().tolist())
            test_proba+=np.array(preds)
        folds_done+=1
        del model; gpu_cleanup()
    return test_proba / max(folds_done * N_TTA, 1)

print('Training infrastructure ready (Focal Loss fixed, OneCycleLR per-group, disk-save, correct TTA).')
print('CELL 9 COMPLETE')

Training infrastructure ready (Focal Loss fixed, OneCycleLR per-group, disk-save, correct TTA).
CELL 9 COMPLETE


In [26]:
# ================================================================
# CELL 10: SigLIP FINE-TUNE (FIXED SPATIAL POOLING)
# ================================================================
from transformers import SiglipModel
gpu_cleanup(); ensure_vram(6.0); set_seeds()

SIG_OOF=CHKPT_DIR/'sig_oof.npy'; SIG_TEST=CHKPT_DIR/'sig_test.npy'; SIG_META=CHKPT_DIR/'sig_meta.json'

class SigLIPFT(nn.Module):
    def __init__(self, vm, d=1152):
        super().__init__()
        self.visual=vm
        self.head=nn.Sequential(nn.LayerNorm(d),nn.Dropout(0.3),nn.Linear(d,256),nn.GELU(),nn.Dropout(0.2),nn.Linear(256,1))
        
    def forward(self,x): 
        # 🚨 FIXED: Bypass the text-aligned pooler to preserve forensic spatial anomalies
        out = self.visual(pixel_values=x).last_hidden_state
        return self.head(out.mean(dim=1).float()).squeeze(-1)

def freeze_sig(m):
    for p in m.visual.parameters(): p.requires_grad=False
def unfreeze_sig(m):
    for p in m.visual.encoder.layers[-2:].parameters(): p.requires_grad=True
    if hasattr(m.visual,'post_layernorm'):
        for p in m.visual.post_layernorm.parameters(): p.requires_grad=True
def get_sig_opt(m):
    bb=[p for p in m.visual.parameters() if p.requires_grad]
    return torch.optim.AdamW([{'params':bb,'lr':5e-6,'weight_decay':0.05},
                               {'params':list(m.head.parameters()),'lr':1e-4,'weight_decay':0.01}])
def make_sig():
    b=SiglipModel.from_pretrained('google/siglip-so400m-patch14-224')
    m=SigLIPFT(b.vision_model); del b; return m

sig_tr_tfm=make_train_tfm(SIGLIP_MEAN,SIGLIP_STD,224)
sig_va_tfm=make_val_tfm(SIGLIP_MEAN,SIGLIP_STD,224)
sig_tta_tfm=make_tta_tfm(SIGLIP_MEAN,SIGLIP_STD,224)

if not FORCE_FRESH and SIG_OOF.exists() and SIG_TEST.exists():
    print('SKIP: Loading SigLIP from checkpoint')
    sig_oof=np.load(SIG_OOF); sig_test=np.load(SIG_TEST); meta=json.load(open(SIG_META))
else:
    sig_oof=np.zeros(len(y_all)); fold_f1s=[]
    print('='*60+'\nSigLIP FINE-TUNE — 5-Fold CV\n'+'='*60)
    for fold,(ti,vi) in enumerate(skf.split(y_all,y_all)):
        gpu_cleanup()
        tr_ds=AlbuDataset([train_paths[i] for i in ti],y_all[ti],sig_tr_tfm)
        va_ds=AlbuDataset([train_paths[i] for i in vi],y_all[vi],sig_va_tfm)
        model=make_sig().to(DEVICE)
        pw=torch.tensor([(len(ti)-y_all[ti].sum())/max(y_all[ti].sum(),1)],device=DEVICE)
        crit=FocalBCEWithLogitsLoss(gamma=2.0, pos_weight=pw)
        f1,vp=train_fold('sig',fold,model,tr_ds,va_ds,crit,freeze_sig,unfreeze_sig,get_sig_opt,batch_size=16)
        sig_oof[vi]=vp; fold_f1s.append(f1)
        del model; gpu_cleanup()
    print('\n-- TTA Inference --')
    sig_test=run_tta_disk('sig',make_sig,sig_tta_tfm,img_size=224,batch_size=32)
    meta={'val_f1_mean':float(np.mean(fold_f1s)),'val_f1_std':float(np.std(fold_f1s)),
          'val_auc_mean':float(roc_auc_score(y_all,sig_oof))}
    np.save(SIG_OOF,sig_oof); np.save(SIG_TEST,sig_test); json.dump(meta,open(SIG_META,'w'))

results_tracker['siglip_ft']={'name':'SigLIP-FT',**meta,'oof_proba':sig_oof.copy()}
oof_store['siglip_ft']=sig_oof.copy(); test_pred_store['siglip_ft']=sig_test.copy()
print(f'SigLIP-FT: F1={meta["val_f1_mean"]:.4f} | AUC={meta["val_auc_mean"]:.4f}')
print('CELL 10 COMPLETE')

  VRAM: 0.0GB used | 25.4GB free
SigLIP FINE-TUNE — 5-Fold CV


Loading weights: 100%|██████████| 888/888 [00:00<00:00, 5666.61it/s]


   Fold 1: S1=0.8685 -> S2=0.8807


Loading weights: 100%|██████████| 888/888 [00:00<00:00, 5369.82it/s]


   Fold 2: S1=0.8872 -> S2=0.8910


...

  TTA Fold 4...


Loading weights: 100%|██████████| 888/888 [00:00<00:00, 5465.97it/s]


  TTA Fold 5...


Loading weights: 100%|██████████| 888/888 [00:00<00:00, 6339.07it/s]


SigLIP-FT: F1=0.8939 | AUC=0.9582
CELL 10 COMPLETE


In [27]:
# ================================================================
# CELL 11: DINOv2-BASE @ 518px FINE-TUNE — V5's STAR (0.9048)
# ================================================================
gpu_cleanup(); ensure_vram(6.0); set_seeds()

DINO_OOF=CHKPT_DIR/'dino_oof.npy'; DINO_TEST=CHKPT_DIR/'dino_test.npy'; DINO_META=CHKPT_DIR/'dino_meta.json'

class DINOv2FT(nn.Module):
    def __init__(self, bb, d=768):
        super().__init__()
        self.backbone=bb
        self.head=nn.Sequential(nn.LayerNorm(d),nn.Dropout(0.3),nn.Linear(d,256),nn.GELU(),nn.Dropout(0.2),nn.Linear(256,1))
    def forward(self,x): return self.head(self.backbone(x)).squeeze(-1)

def freeze_dino(m):
    for p in m.backbone.parameters(): p.requires_grad=False
def unfreeze_dino(m):
    for p in m.backbone.blocks[-3:].parameters(): p.requires_grad=True
    if hasattr(m.backbone,'norm'):
        for p in m.backbone.norm.parameters(): p.requires_grad=True
def get_dino_opt(m):
    bb=[p for p in m.backbone.parameters() if p.requires_grad]
    return torch.optim.AdamW([{'params':bb,'lr':5e-6,'weight_decay':0.05},
                               {'params':list(m.head.parameters()),'lr':1e-4,'weight_decay':0.01}])
def make_dino():
    return DINOv2FT(timm.create_model('vit_base_patch14_dinov2',pretrained=True,num_classes=0))

dino_tr_tfm=make_train_tfm(DINO_MEAN,DINO_STD,518)
dino_va_tfm=make_val_tfm(DINO_MEAN,DINO_STD,518)
dino_tta_tfm=make_tta_tfm(DINO_MEAN,DINO_STD,518)

if not FORCE_FRESH and DINO_OOF.exists() and DINO_TEST.exists():
    print('SKIP: Loading DINOv2-B from checkpoint')
    dino_oof=np.load(DINO_OOF); dino_test=np.load(DINO_TEST); meta=json.load(open(DINO_META))
else:
    dino_oof=np.zeros(len(y_all)); fold_f1s=[]
    print('='*60+'\nDINOv2-BASE @ 518px FINE-TUNE — 5-Fold CV\n'+'='*60)
    for fold,(ti,vi) in enumerate(skf.split(y_all,y_all)):
        gpu_cleanup()
        tr_ds=AlbuDataset([train_paths[i] for i in ti],y_all[ti],dino_tr_tfm,img_size=518)
        va_ds=AlbuDataset([train_paths[i] for i in vi],y_all[vi],dino_va_tfm,img_size=518)
        model=make_dino().to(DEVICE)
        pw=torch.tensor([(len(ti)-y_all[ti].sum())/max(y_all[ti].sum(),1)],device=DEVICE)
        crit=FocalBCEWithLogitsLoss(gamma=2.0, pos_weight=pw)
        f1,vp=train_fold('dino',fold,model,tr_ds,va_ds,crit,freeze_dino,unfreeze_dino,get_dino_opt,batch_size=16)
        dino_oof[vi]=vp; fold_f1s.append(f1)
        del model; gpu_cleanup()
    print('\n-- TTA Inference --')
    dino_test=run_tta_disk('dino',make_dino,dino_tta_tfm,img_size=518,batch_size=16)
    meta={'val_f1_mean':float(np.mean(fold_f1s)),'val_f1_std':float(np.std(fold_f1s)),
          'val_auc_mean':float(roc_auc_score(y_all,dino_oof))}
    np.save(DINO_OOF,dino_oof); np.save(DINO_TEST,dino_test); json.dump(meta,open(DINO_META,'w'))

results_tracker['dino_ft']={'name':'DINOv2-B-FT',**meta,'oof_proba':dino_oof.copy()}
oof_store['dino_ft']=dino_oof.copy(); test_pred_store['dino_ft']=dino_test.copy()
print(f'DINOv2-B-FT: F1={meta["val_f1_mean"]:.4f} | AUC={meta["val_auc_mean"]:.4f}')
print('CELL 11 COMPLETE')

  VRAM: 0.1GB used | 25.4GB free
DINOv2-BASE @ 518px FINE-TUNE — 5-Fold CV
   Fold 1: S1=0.8139 -> S2=0.8997
   Fold 2: S1=0.8280 -> S2=0.9046
   Fold 3: S1=0.8185 -> S2=0.8905
   Fold 4: S1=0.8464 -> S2=0.9122
   Fold 5: S1=0.8112 -> S2=0.9076

-- TTA Inference --
  TTA Fold 1...
  TTA Fold 2...
  TTA Fold 3...
  TTA Fold 4...
  TTA Fold 5...
DINOv2-B-FT: F1=0.9029 | AUC=0.9627
CELL 11 COMPLETE


In [28]:
# ================================================================
# CELL 12: CLIP ViT-L/14 FINE-TUNE — 5-Fold CV
# ================================================================
gpu_cleanup(); ensure_vram(6.0); set_seeds()

CLIP_OOF=CHKPT_DIR/'clip_oof.npy'; CLIP_TEST=CHKPT_DIR/'clip_test.npy'; CLIP_META=CHKPT_DIR/'clip_meta.json'

class CLIPFT(nn.Module):
    def __init__(self, vis, d=1024): # 🚨 Changed to 1024 (Raw ViT-L width)
        super().__init__()
        self.visual=vis
        self.head=nn.Sequential(nn.LayerNorm(d),nn.Dropout(0.3),nn.Linear(d,256),nn.GELU(),nn.Dropout(0.2),nn.Linear(256,1))
        
    def forward(self, x): 
        # 🚨 Bypass the text-projection matrix to preserve raw forensic data
        x = self.visual.conv1(x)  # shape = [*, width, grid, grid]
        x = x.reshape(x.shape[0], x.shape[1], -1)  # shape = [*, width, grid ** 2]
        x = x.permute(0, 2, 1)  # shape = [*, grid ** 2, width]
        x = torch.cat([self.visual.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device), x], dim=1)
        x = x + self.visual.positional_embedding.to(x.dtype)
        x = self.visual.ln_pre(x)

        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.visual.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD

        # We take the CLS token and apply post-LayerNorm, BUT WE DO NOT MULTIPLY BY self.proj!
        out = self.visual.ln_post(x[:, 0, :])
        
        return self.head(out.float()).squeeze(-1)

def freeze_clip(m):
    for p in m.visual.parameters(): p.requires_grad=False
def unfreeze_clip(m):
    for p in m.visual.transformer.resblocks[-2:].parameters(): p.requires_grad=True
    if hasattr(m.visual,'ln_post'):
        for p in m.visual.ln_post.parameters(): p.requires_grad=True
def get_clip_opt(m):
    bb=[p for p in m.visual.parameters() if p.requires_grad]
    return torch.optim.AdamW([{'params':bb,'lr':5e-6,'weight_decay':0.05},
                               {'params':list(m.head.parameters()),'lr':1e-4,'weight_decay':0.01}])
def make_clip():
    cm,_=clip.load('ViT-L/14',device='cpu')
    return CLIPFT(cm.float().visual)

clip_tr_tfm=make_train_tfm(CLIP_MEAN,CLIP_STD,224)
clip_va_tfm=make_val_tfm(CLIP_MEAN,CLIP_STD,224)
clip_tta_tfm=make_tta_tfm(CLIP_MEAN,CLIP_STD,224)

if not FORCE_FRESH and CLIP_OOF.exists() and CLIP_TEST.exists():
    print('SKIP: Loading CLIP from checkpoint')
    clip_oof=np.load(CLIP_OOF); clip_test=np.load(CLIP_TEST); meta=json.load(open(CLIP_META))
else:
    clip_oof=np.zeros(len(y_all)); fold_f1s=[]
    print('='*60+'\nCLIP ViT-L/14 FINE-TUNE — 5-Fold CV\n'+'='*60)
    for fold,(ti,vi) in enumerate(skf.split(y_all,y_all)):
        gpu_cleanup()
        tr_ds=AlbuDataset([train_paths[i] for i in ti],y_all[ti],clip_tr_tfm)
        va_ds=AlbuDataset([train_paths[i] for i in vi],y_all[vi],clip_va_tfm)
        model=make_clip().to(DEVICE)
        pw=torch.tensor([(len(ti)-y_all[ti].sum())/max(y_all[ti].sum(),1)],device=DEVICE)
        crit=FocalBCEWithLogitsLoss(gamma=2.0, pos_weight=pw)
        f1,vp=train_fold('clip',fold,model,tr_ds,va_ds,crit,freeze_clip,unfreeze_clip,get_clip_opt,batch_size=16)
        clip_oof[vi]=vp; fold_f1s.append(f1)
        del model; gpu_cleanup()
    print('\n-- TTA Inference --')
    clip_test=run_tta_disk('clip',make_clip,clip_tta_tfm,img_size=224,batch_size=32)
    meta={'val_f1_mean':float(np.mean(fold_f1s)),'val_f1_std':float(np.std(fold_f1s)),
          'val_auc_mean':float(roc_auc_score(y_all,clip_oof))}
    np.save(CLIP_OOF,clip_oof); np.save(CLIP_TEST,clip_test); json.dump(meta,open(CLIP_META,'w'))

results_tracker['clip_ft']={'name':'CLIP-FT',**meta,'oof_proba':clip_oof.copy()}
oof_store['clip_ft']=clip_oof.copy(); test_pred_store['clip_ft']=clip_test.copy()
print(f'CLIP-FT: F1={meta["val_f1_mean"]:.4f} | AUC={meta["val_auc_mean"]:.4f}')
print('CELL 12 COMPLETE')

  VRAM: 0.1GB used | 25.4GB free
CLIP ViT-L/14 FINE-TUNE — 5-Fold CV
   Fold 1: S1=0.8554 -> S2=0.8742
   Fold 2: S1=0.8847 -> S2=0.9022
   Fold 3: S1=0.8762 -> S2=0.8988
   Fold 4: S1=0.8903 -> S2=0.9044
   Fold 5: S1=0.8865 -> S2=0.9003

-- TTA Inference --
  TTA Fold 1...
  TTA Fold 2...
  TTA Fold 3...
  TTA Fold 4...
  TTA Fold 5...
CLIP-FT: F1=0.8960 | AUC=0.9600
CELL 12 COMPLETE


In [29]:
# ================================================================
# CELL 13: XGBoost on Forensic + CNN
# ================================================================
set_seeds()
vt=VarianceThreshold(threshold=1e-10)
for_tr=vt.fit_transform(forensic_train); for_te=vt.transform(forensic_test)
X_tr=np.hstack([for_tr,cnn_train]); X_te=np.hstack([for_te,cnn_test])
print(f'XGB input: {X_tr.shape}')

XGB_P=dict(n_estimators=500,max_depth=4,learning_rate=0.05,subsample=0.7,
           colsample_bytree=0.7,min_child_weight=5,reg_alpha=0.1,reg_lambda=1.0,
           gamma=0.1,eval_metric='logloss',random_state=SEED,tree_method='hist',device='cuda')

xgb_oof=np.zeros(len(y_all)); fold_f1s=[]
for fold,(ti,vi) in enumerate(skf.split(y_all,y_all)):
    sc=RobustScaler().fit(X_tr[ti]); pca=PCA(n_components=min(128,X_tr.shape[1]-1),random_state=SEED)
    Xtr_t=pca.fit_transform(sc.transform(X_tr[ti])); Xv_t=pca.transform(sc.transform(X_tr[vi]))
    clf=xgb.XGBClassifier(**XGB_P); clf.fit(Xtr_t,y_all[ti].astype(int))
    vp=clf.predict_proba(Xv_t)[:,1]; xgb_oof[vi]=vp
    fold_f1s.append(f1_score(y_all[vi].astype(int),(vp>=0.5).astype(int)))

sc_all=RobustScaler().fit(X_tr); pca_all=PCA(n_components=min(128,X_tr.shape[1]-1),random_state=SEED)
Xtr_all=pca_all.fit_transform(sc_all.transform(X_tr))
clf_all=xgb.XGBClassifier(**XGB_P); clf_all.fit(Xtr_all,y_all.astype(int))
xgb_test=clf_all.predict_proba(pca_all.transform(sc_all.transform(X_te)))[:,1]

results_tracker['xgb']={'name':'XGB','val_f1_mean':float(np.mean(fold_f1s)),
    'val_f1_std':float(np.std(fold_f1s)),'val_auc_mean':float(roc_auc_score(y_all,xgb_oof)),'oof_proba':xgb_oof.copy()}
oof_store['xgb']=xgb_oof.copy(); test_pred_store['xgb']=xgb_test.copy()
print(f'XGB: F1={np.mean(fold_f1s):.4f} | AUC={roc_auc_score(y_all,xgb_oof):.4f}')
print('CELL 13 COMPLETE')

XGB input: (4800, 1367)
XGB: F1=0.7510 | AUC=0.8319
CELL 13 COMPLETE


In [30]:
# ================================================================
# CELL 14: ISOTONIC CALIBRATION -> HILL CLIMBING -> ENSEMBLE
# Proven technique from V10 GOAT (achieved 0.9249)
# ================================================================
set_seeds()
all_keys = ['siglip_ft','clip_ft','dino_ft','xgb']
available = [k for k in all_keys if k in oof_store]

print('='*65)
print('MODEL SCORES')
print('='*65)
for k in available:
    r=results_tracker[k]
    print(f'  {r["name"]:<20} F1={r["val_f1_mean"]:.4f} AUC={r["val_auc_mean"]:.4f}')

# ── Stage 1: Isotonic Calibration ──
print('\n--- ISOTONIC CALIBRATION ---')
cal_oof={}; cal_test={}
for k in available:
    ir=IsotonicRegression(out_of_bounds='clip')
    cal_oof[k]=ir.fit_transform(oof_store[k],y_all)
    cal_test[k]=ir.transform(test_pred_store[k])
    print(f'  {k:<15} Raw={oof_store[k].mean():.3f} -> Cal={cal_oof[k].mean():.3f}')

# ── Stage 2: Caruana's Hill Climbing ──
print('\n--- HILL CLIMBING (50 steps) ---')
weights={k:0 for k in available}
blend=np.zeros(len(y_all)); best_hc_f1=0
for step in range(1,51):
    best_k=None; best_sf1=0; best_st=0.5
    for k in available:
        tb=(blend*(step-1)+cal_oof[k])/step
        for t in np.arange(0.35,0.65,0.01):
            f=f1_score(y_all,(tb>=t).astype(int))
            if f>best_sf1: best_sf1=f; best_k=k; best_st=t
    weights[best_k]+=1
    blend=(blend*(step-1)+cal_oof[best_k])/step
    if step%10==0 or step==1:
        print(f'  Step {step:<2} | Added: {best_k:<15} | F1: {best_sf1:.4f} @ thr={best_st:.2f}')
    best_hc_f1=best_sf1

# Find optimal OOF threshold
best_oof_thr=0.5; best_oof_f1=0
for t in np.arange(0.35,0.65,0.01):
    f=f1_score(y_all,(blend>=t).astype(int))
    if f>best_oof_f1: best_oof_f1=f; best_oof_thr=round(t,2)

print(f'\n--- HILL CLIMBING WEIGHTS ---')
tw=sum(weights.values())
for k in available: print(f'  {k:<15}: {weights[k]}/{tw} ({weights[k]/tw*100:.1f}%)')
print(f'\nOOF F1 = {best_oof_f1:.4f} @ thr = {best_oof_thr}')

# ── Stage 3: Build test predictions with HC weights ──
test_blend=np.zeros(len(test_paths))
for k in available:
    w=weights[k]/tw
    test_blend+=w*cal_test[k]

# Also try simple strategies for comparison
print('\n--- COMPARISON ---')
# A: Simple weighted avg (no calibration)
f1sq={k:results_tracker[k]['val_f1_mean']**2 for k in available}
twA=sum(f1sq.values())
oof_A=sum(f1sq[k]/twA*oof_store[k] for k in available)
bf_A=0; bt_A=0.5
for t in np.arange(0.35,0.65,0.01):
    f=f1_score(y_all,(oof_A>=t).astype(int))
    if f>bf_A: bf_A=f; bt_A=round(t,2)
print(f'  Simple weighted avg: F1={bf_A:.4f} @ thr={bt_A}')
print(f'  Hill climbing:       F1={best_oof_f1:.4f} @ thr={best_oof_thr}')

# Use whichever is best
if bf_A > best_oof_f1:
    print('\nSimple avg WINS')
    BEST_THR=bt_A; final_oof=oof_A
    test_final=sum(f1sq[k]/twA*test_pred_store[k] for k in available)
    best_final_f1=bf_A; method='weighted_avg'
else:
    print('\nHill climbing WINS')
    BEST_THR=best_oof_thr; final_oof=blend
    test_final=test_blend; best_final_f1=best_oof_f1; method='hill_climb'

results_tracker['ensemble']={'name':f'Ensemble({method})','val_f1_mean':best_final_f1,
    'val_f1_std':0.0,'val_auc_mean':float(roc_auc_score(y_all,final_oof)),'oof_proba':final_oof.copy()}
oof_store['ensemble']=final_oof.copy()
test_pred_store['ensemble']=test_final.copy()

print(f'\n{"="*65}')
print(f'FINAL: {method} | OOF F1={best_final_f1:.4f} | Thr={BEST_THR}')
print(f'AUC={roc_auc_score(y_all,final_oof):.4f}')
print('CELL 14 COMPLETE')

MODEL SCORES
  SigLIP-FT            F1=0.8939 AUC=0.9582
  CLIP-FT              F1=0.8960 AUC=0.9600
  DINOv2-B-FT          F1=0.9029 AUC=0.9627
  XGB                  F1=0.7510 AUC=0.8319

--- ISOTONIC CALIBRATION ---
  siglip_ft       Raw=0.506 -> Cal=0.482
  clip_ft         Raw=0.494 -> Cal=0.482
  dino_ft         Raw=0.501 -> Cal=0.482
  xgb             Raw=0.489 -> Cal=0.482

--- HILL CLIMBING (50 steps) ---
  Step 1  | Added: dino_ft         | F1: 0.9048 @ thr=0.35
  Step 10 | Added: dino_ft         | F1: 0.9310 @ thr=0.47
  Step 20 | Added: xgb             | F1: 0.9325 @ thr=0.47
  Step 30 | Added: siglip_ft       | F1: 0.9323 @ thr=0.47
  Step 40 | Added: xgb             | F1: 0.9325 @ thr=0.47
  Step 50 | Added: dino_ft         | F1: 0.9325 @ thr=0.47

--- HILL CLIMBING WEIGHTS ---
  siglip_ft      : 12/50 (24.0%)
  clip_ft        : 10/50 (20.0%)
  dino_ft        : 18/50 (36.0%)
  xgb            : 10/50 (20.0%)

OOF F1 = 0.9325 @ thr = 0.47

--- COMPARISON ---
  Simple weighte

In [31]:
# ================================================================
# CELL 15: ANALYSIS
# ================================================================
show=['siglip_ft','dino_ft','clip_ft','xgb','ensemble']
print('='*70)
print(f'{"Model":<22} {"F1":>8} {"Std":>6} {"AUC":>8}')
print('='*70)
for k in show:
    if k not in results_tracker: continue
    r=results_tracker[k]
    print(f'  {r["name"]:<20} {r["val_f1_mean"]:>8.4f} {r.get("val_f1_std",0):>6.4f} {r["val_auc_mean"]:>8.4f}')

corr_keys=[k for k in ['siglip_ft','dino_ft','clip_ft','xgb'] if k in oof_store]
if len(corr_keys)>=2:
    cd=np.column_stack([oof_store[k] for k in corr_keys])
    cm=np.corrcoef(cd.T)
    print('\nCorrelation:')
    lbls=[results_tracker[k]['name'][:12] for k in corr_keys]
    print(f'{"":>14}','  '.join(f'{l:>12}' for l in lbls))
    for i,l in enumerate(lbls): print(f'{l:>14}','  '.join(f'{cm[i,j]:>12.4f}' for j in range(len(lbls))))

oof_p=(final_oof>=BEST_THR).astype(int)
cm_mat=confusion_matrix(y_all.astype(int),oof_p)
fig,axes=plt.subplots(1,2,figsize=(12,5))
sns.heatmap(cm_mat,annot=True,fmt='d',cmap='Blues',ax=axes[0],xticklabels=['Real','AI'],yticklabels=['Real','AI'])
axes[0].set_title(f'CM (thr={BEST_THR})')
for k in show:
    if k not in oof_store: continue
    fpr,tpr,_=roc_curve(y_all,oof_store[k])
    axes[1].plot(fpr,tpr,label=f'{results_tracker[k]["name"][:14]} ({roc_auc_score(y_all,oof_store[k]):.3f})')
axes[1].plot([0,1],[0,1],'k--',alpha=0.3); axes[1].legend(fontsize=7)
axes[1].set_title('ROC'); plt.tight_layout(); plt.savefig('analysis_v11.png',dpi=150); plt.show()

print(f'\nAccuracy:  {accuracy_score(y_all.astype(int),oof_p):.4f}')
print(f'Precision: {precision_score(y_all.astype(int),oof_p):.4f}')
print(f'Recall:    {recall_score(y_all.astype(int),oof_p):.4f}')
print(f'F1:        {best_final_f1:.4f}')
print(f'AUC:       {roc_auc_score(y_all,final_oof):.4f}')
print('CELL 15 COMPLETE')

Model                        F1    Std      AUC
  SigLIP-FT              0.8939 0.0088   0.9582
  DINOv2-B-FT            0.9029 0.0074   0.9627
  CLIP-FT                0.8960 0.0111   0.9600
  XGB                    0.7510 0.0066   0.8319
  Ensemble(hill_climb)   0.9325 0.0000   0.9796

Correlation:
                  SigLIP-FT   DINOv2-B-FT       CLIP-FT           XGB
     SigLIP-FT       1.0000        0.8415        0.8710        0.6226
   DINOv2-B-FT       0.8415        1.0000        0.8415        0.6376
       CLIP-FT       0.8710        0.8415        1.0000        0.6233
           XGB       0.6226        0.6376        0.6233        1.0000

Accuracy:  0.9340
Precision: 0.9197
Recall:    0.9456
F1:        0.9325
AUC:       0.9796
CELL 15 COMPLETE


In [32]:
# ================================================================
# CELL 16: FINAL SUBMISSION
# ================================================================
test_proba=test_pred_store['ensemble']
thr=BEST_THR

# Check if single model beats ensemble
for k in ['dino_ft','siglip_ft','clip_ft']:
    if k in results_tracker and results_tracker[k]['val_f1_mean']>best_final_f1:
        print(f'FALLBACK: {k} ({results_tracker[k]["val_f1_mean"]:.4f}) > ensemble')
        test_proba=test_pred_store[k]
        so=oof_store[k]; bf=0
        for t in np.arange(0.35,0.65,0.01):
            f=f1_score(y_all,(so>=t).astype(int))
            if f>bf: bf=f; thr=round(t,2)
        best_final_f1=bf; break

preds=(test_proba>=thr).astype(int)
sub=pd.DataFrame({'image_id':df_test['image_id'].values,'ground_truth':preds})
assert sub.shape==(len(df_test),2)
assert sub['ground_truth'].isin([0,1]).all()

n0=(sub['ground_truth']==0).sum(); n1=(sub['ground_truth']==1).sum()
print('='*60)
print(f'FINAL: Val F1={best_final_f1:.4f} | Thr={thr}')
print(f'Predictions: Real={n0} ({n0/len(sub):.1%}) | AI={n1} ({n1/len(sub):.1%})')
print('='*60)
print(sub.head())

for p in ['./submission.csv','/home/jovyan/work/data/submission.csv']:
    try:
        os.makedirs(os.path.dirname(p) if os.path.dirname(p) else '.',exist_ok=True)
        sub.to_csv(p,index=False); print(f'Saved: {p}'); break
    except: continue

print('VERSION 11 COMPLETE')

FINAL: Val F1=0.9325 | Thr=0.47
Predictions: Real=1033 (50.2%) | AI=1025 (49.8%)
                                   image_id  ground_truth
0  3ecf1af5-6a8f-416a-9b4c-df9f2e0a0a80.jpg             1
1  2789b3fe-a337-4dc2-b42c-8bccde1f68fb.jpg             0
2  01a342c6-c3fc-4b55-8c22-13c1a556ba87.jpg             0
3  ac784910-b461-498d-b3a8-50b1e4116b11.jpg             0
4  6dcd4df6-7447-4bcf-a29b-f7f53b4c3ed4.jpg             0
Saved: ./submission.csv
VERSION 11 COMPLETE


In [34]:
# ================================================================
# CELL 17: ACADEMIC PAPER VISUALIZATION SUITE
# ================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, average_precision_score, 
                             f1_score, precision_score, recall_score, brier_score_loss)
from sklearn.calibration import calibration_curve

# Setup Academic Formatting
sns.set_theme(style="whitegrid", context="paper", font_scale=1.3)
plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold', 'figure.titleweight': 'bold'})

IMG_DIR = Path('./Images_v11')
IMG_DIR.mkdir(exist_ok=True)

# 1. Prepare Data
y_true = y_all
models = ['dino_ft', 'siglip_ft', 'clip_ft', 'xgb']
model_names = {'dino_ft': 'DINOv2-BASE', 'siglip_ft': 'SigLIP-400M', 'clip_ft': 'CLIP-L/14', 'xgb': 'XGBoost'}
colors = {'dino_ft': '#e74c3c', 'siglip_ft': '#3498db', 'clip_ft': '#2ecc71', 'xgb': '#f39c12', 'ensemble': '#9b59b6'}

# Recreate Ensemble Probabilities (Using your Hill Climbing Weights)
weights = {'dino_ft': 0.36, 'siglip_ft': 0.24, 'clip_ft': 0.20, 'xgb': 0.20}
ens_proba = np.zeros_like(y_true, dtype=float)
for m in models:
    ens_proba += oof_store[m] * weights[m]

print("Generating Academic Figures...")

# ---------------------------------------------------------
# 1. Model Confidence & Separation Margin (THE MOST IMPORTANT)
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.kdeplot(ens_proba[y_true == 0], fill=True, color='#2ecc71', label='True Class: Real (0)', alpha=0.6, linewidth=2)
sns.kdeplot(ens_proba[y_true == 1], fill=True, color='#e74c3c', label='True Class: Deepfake (1)', alpha=0.6, linewidth=2)
plt.axvline(x=0.47, color='k', linestyle='--', label='Optimal Threshold (0.47)')
plt.title('Ensemble Confidence & Separation Margin\n(Notice the massive separation gap)')
plt.xlabel('Predicted Probability of Deepfake')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
#plt.savefig(IMG_DIR / '01_Separation_Margin.pdf', dpi=300)
plt.savefig(IMG_DIR / '01_Separation_Margin.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 2. ROC Curve
# ---------------------------------------------------------
plt.figure(figsize=(8, 8))
for m in models:
    fpr, tpr, _ = roc_curve(y_true, oof_store[m])
    plt.plot(fpr, tpr, label=f'{model_names[m]} (AUC = {auc(fpr, tpr):.4f})', color=colors[m], linewidth=2)
fpr, tpr, _ = roc_curve(y_true, ens_proba)
plt.plot(fpr, tpr, label=f'Proposed Ensemble (AUC = {auc(fpr, tpr):.4f})', color=colors['ensemble'], linewidth=3, linestyle='--')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.title('Receiver Operating Characteristic (ROC)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
#plt.savefig(IMG_DIR / '02_ROC_Curve.pdf', dpi=300)
plt.savefig(IMG_DIR / '02_ROC_Curve.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 3. Precision-Recall Curve (PRC)
# ---------------------------------------------------------
plt.figure(figsize=(8, 8))
for m in models:
    p, r, _ = precision_recall_curve(y_true, oof_store[m])
    plt.plot(r, p, label=f'{model_names[m]} (AP = {average_precision_score(y_true, oof_store[m]):.4f})', color=colors[m], linewidth=2)
p, r, _ = precision_recall_curve(y_true, ens_proba)
plt.plot(r, p, label=f'Proposed Ensemble (AP = {average_precision_score(y_true, ens_proba):.4f})', color=colors['ensemble'], linewidth=3, linestyle='--')
plt.title('Precision-Recall Curve (PRC)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(loc='lower left')
plt.tight_layout()
#plt.savefig(IMG_DIR / '03_PR_Curve.pdf', dpi=300)
plt.savefig(IMG_DIR / '03_PR_Curve.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 4. Performance Metrics Across Thresholds
# ---------------------------------------------------------
thresholds = np.linspace(0.1, 0.9, 100)
f1s = [f1_score(y_true, ens_proba >= t) for t in thresholds]
precs = [precision_score(y_true, ens_proba >= t, zero_division=0) for t in thresholds]
recs = [recall_score(y_true, ens_proba >= t) for t in thresholds]

plt.figure(figsize=(10, 6))
plt.plot(thresholds, f1s, label='F1 Score', color='purple', linewidth=3)
plt.plot(thresholds, precs, label='Precision', color='blue', linestyle='-.', linewidth=2)
plt.plot(thresholds, recs, label='Recall', color='green', linestyle=':', linewidth=2)
plt.axvline(x=0.47, color='red', linestyle='--', label='Max F1 Threshold (0.47)')
plt.title('Ensemble Performance Metrics vs. Decision Threshold')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.legend()
plt.tight_layout()
#plt.savefig(IMG_DIR / '04_Metrics_vs_Threshold.pdf', dpi=300)
plt.savefig(IMG_DIR / '04_Metrics_vs_Threshold.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 5. Greedy Forward Selection (Hill Climbing Trajectory)
# ---------------------------------------------------------
# Re-simulating a quick 20-step trajectory for visualization
current_preds = np.zeros_like(y_true, dtype=float)
step_f1s = []
for step in range(1, 21):
    best_f1 = 0; best_m = None
    for m in models:
        test_preds = (current_preds * (step - 1) + oof_store[m]) / step
        f1 = np.max([f1_score(y_true, test_preds >= t) for t in np.linspace(0.3, 0.6, 31)])
        if f1 > best_f1: best_f1 = f1; best_m = m
    current_preds = (current_preds * (step - 1) + oof_store[best_m]) / step
    step_f1s.append(best_f1)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 21), step_f1s, marker='o', color='#9b59b6', linewidth=2, markersize=8)
plt.axhline(y=np.max(step_f1s), color='r', linestyle='--', label=f'Peak F1 ({np.max(step_f1s):.4f})')
plt.title('Greedy Forward Selection (Hill Climbing) Trajectory')
plt.xlabel('Ensemble Step (Models Added)')
plt.ylabel('Validation F1 Score')
plt.xticks(range(1, 21))
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
#plt.savefig(IMG_DIR / '05_Hill_Climbing_Trajectory.pdf', dpi=300)
plt.savefig(IMG_DIR / '05_Hill_Climbing_Trajectory.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 6. Ensemble Weight Composition (Pie Chart)
# ---------------------------------------------------------
plt.figure(figsize=(8, 8))
explode = (0.05, 0, 0, 0)  # Highlight DINO
plt.pie(weights.values(), labels=[model_names[k] for k in weights.keys()], autopct='%1.1f%%', 
        startangle=140, colors=[colors[k] for k in weights.keys()], explode=explode, shadow=True,
        textprops={'fontweight': 'bold', 'fontsize': 14})
plt.title('Final Ensemble Weight Composition')
plt.tight_layout()
#plt.savefig(IMG_DIR / '06_Weight_Composition.pdf', dpi=300)
plt.savefig(IMG_DIR / '06_Weight_Composition.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 7. Correlation Heatmap
# ---------------------------------------------------------
pred_df = pd.DataFrame({model_names[m]: oof_store[m] for m in models})
corr = pred_df.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=0.5, vmax=1.0, fmt=".3f", linewidths=1, linecolor='black')
plt.title('Model Prediction Correlation Matrix\n(Shows why XGBoost is highly valuable)')
plt.tight_layout()
#plt.savefig(IMG_DIR / '07_Correlation_Heatmap.pdf', dpi=300)
plt.savefig(IMG_DIR / '07_Correlation_Heatmap.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 8. Calibration Curves (Reliability Diagram)
# ---------------------------------------------------------
plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
for m in models:
    prob_true, prob_pred = calibration_curve(y_true, oof_store[m], n_bins=10)
    plt.plot(prob_pred, prob_true, "s-", label=f'{model_names[m]}', color=colors[m], alpha=0.7)
prob_true_ens, prob_pred_ens = calibration_curve(y_true, ens_proba, n_bins=10)
plt.plot(prob_pred_ens, prob_true_ens, "o-", label="Proposed Ensemble", color=colors['ensemble'], linewidth=3)
plt.title('Calibration Curves (Reliability Diagram)')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend(loc="lower right")
plt.tight_layout()
#plt.savefig(IMG_DIR / '08_Calibration_Curve.pdf', dpi=300)
plt.savefig(IMG_DIR / '08_Calibration_Curve.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 9. Error Overlap (Agreement Matrix for False Predictions)
# ---------------------------------------------------------
# Calculate where models made errors (using best individual thresholds ~0.5)
errors = {m: (oof_store[m] >= 0.5) != y_true for m in models}
error_df = pd.DataFrame(errors)
# Co-occurrence of errors
error_cooccurrence = error_df.astype(int).T.dot(error_df.astype(int))

plt.figure(figsize=(8, 6))
sns.heatmap(error_cooccurrence, annot=True, fmt='d', cmap='Reds', xticklabels=[model_names[m] for m in models], yticklabels=[model_names[m] for m in models])
plt.title('Error Co-occurrence Matrix\n(Number of shared misclassifications)')
plt.tight_layout()
#plt.savefig(IMG_DIR / '09_Error_Overlap.pdf', dpi=300)
plt.savefig(IMG_DIR / '09_Error_Overlap.png', dpi=300)
plt.close()

# ---------------------------------------------------------
# 10. Accuracy vs. Computational Cost
# ---------------------------------------------------------
# Approximated parameters (in Millions)
params = {'dino_ft': 86, 'siglip_ft': 400, 'clip_ft': 428, 'xgb': 1, 'ensemble': 915}
f1_scores = {'dino_ft': 0.9029, 'siglip_ft': 0.8939, 'clip_ft': 0.8960, 'xgb': 0.7510, 'ensemble': 0.9325}

plt.figure(figsize=(10, 6))
for m in params.keys():
    c = colors[m] if m != 'ensemble' else colors['ensemble']
    name = model_names.get(m, 'Proposed Ensemble')
    s = 400 if m == 'ensemble' else 200
    marker = '*' if m == 'ensemble' else 'o'
    plt.scatter(params[m], f1_scores[m], color=c, s=s, label=name, marker=marker, edgecolors='black')
    plt.annotate(name, (params[m], f1_scores[m]), textcoords="offset points", xytext=(0,15), ha='center', fontweight='bold')

plt.title('Performance vs. Parameter Count (Efficiency Frontier)')
plt.xlabel('Number of Parameters (Millions)')
plt.ylabel('Validation F1 Score')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xscale('log') # Log scale because XGBoost is tiny and ViTs are huge
plt.tight_layout()
#plt.savefig(IMG_DIR / '10_Accuracy_vs_Computation.pdf', dpi=300)
plt.savefig(IMG_DIR / '10_Accuracy_vs_Computation.png', dpi=300)
plt.close()

print(f"✅ Successfully generated and saved 10 Academic Quality Graphs to {IMG_DIR.absolute()}")

Generating Academic Figures...
✅ Successfully generated and saved 10 Academic Quality Graphs to /home/jovyan/work/data/Images_v11


In [36]:
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, brier_score_loss, confusion_matrix)

print("==========================================================================")
print("      🔬 COMPREHENSIVE ACADEMIC METRICS (VERSION 11 FINAL) 🔬")
print("==========================================================================\n")

# Define V11 Models based on your architecture
v11_models = ['dino_ft', 'siglip_ft', 'clip_ft', 'xgb']
model_names = {
    'dino_ft': 'DINOv2-BASE @ 518px',
    'siglip_ft': 'SigLIP-400M @ 224px',
    'clip_ft': 'CLIP ViT-L/14 @ 224px',
    'xgb': 'XGBoost (Offline Features)'
}

# --- 1. INDIVIDUAL MODEL METRICS ---
print("--- 1. INDIVIDUAL MODEL PERFORMANCE (AT OPTIMAL THRESHOLD) ---")
print(f"{'Model Name':<26} | {'Thr'}  | {'Acc':<6} | {'Prec':<6} | {'Rec':<6} | {'F1':<6} | {'AUC':<6}")
print("-" * 80)

for k in v11_models:
    if k not in oof_store:
        continue
    preds = oof_store[k]
    auc_val = roc_auc_score(y_all, preds)
    
    # Find optimal threshold for F1
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.3, 0.71, 0.01):
        f = f1_score(y_all, (preds >= t).astype(int))
        if f > best_f1: 
            best_f1 = f; best_t = t
            
    # Calculate metrics at optimal threshold
    bin_preds = (preds >= best_t).astype(int)
    acc = accuracy_score(y_all, bin_preds)
    prec = precision_score(y_all, bin_preds, zero_division=0)
    rec = recall_score(y_all, bin_preds)
    
    print(f"{model_names.get(k, k):<26} | {best_t:.2f} | {acc:.4f} | {prec:.4f} | {rec:.4f} | {best_f1:.4f} | {auc_val:.4f}")

# --- 2. ENSEMBLE RECONSTRUCTION ---
print("\n--- 2. FINAL ENSEMBLE METRICS (CALIBRATED + HILL CLIMB) ---")
# Using the exact Hill Climbing weights from your V11 logs
weights = {'dino_ft': 0.36, 'siglip_ft': 0.24, 'clip_ft': 0.20, 'xgb': 0.20} 

ens_proba = np.zeros_like(y_all, dtype=float)
for m in v11_models:
    if m in oof_store:
        # Use calibrated probabilities if they exist in memory, otherwise fallback to raw
        prob = calibrated_oof[m] if 'calibrated_oof' in globals() and m in calibrated_oof else oof_store[m]
        ens_proba += prob * weights.get(m, 0)

# Find optimal threshold for ensemble
ens_best_f1, ens_best_t = 0, 0.47
for t in np.arange(0.35, 0.65, 0.01):
    f1 = f1_score(y_all, (ens_proba >= t).astype(int))
    if f1 > ens_best_f1: 
        ens_best_f1 = f1; ens_best_t = t

bin_ens_preds = (ens_proba >= ens_best_t).astype(int)

# Calculate in-depth ensemble metrics
ens_acc = accuracy_score(y_all, bin_ens_preds)
ens_prec = precision_score(y_all, bin_ens_preds, zero_division=0)
ens_rec = recall_score(y_all, bin_ens_preds)
ens_auc = roc_auc_score(y_all, ens_proba)
ens_brier = brier_score_loss(y_all, ens_proba)
tn, fp, fn, tp = confusion_matrix(y_all, bin_ens_preds).ravel()

print(f"Accuracy   : {ens_acc:.4f} (Overall correctness across both classes)")
print(f"Precision  : {ens_prec:.4f} (When it says AI, it is actually AI this % of the time)")
print(f"Recall     : {ens_rec:.4f} (It successfully caught this % of all AI images)")
print(f"F1 Score   : {ens_best_f1:.4f} (The harmonic mean of Precision & Recall) @ Thr={ens_best_t:.2f}")
print(f"ROC AUC    : {ens_auc:.4f} (Overall class separation capability)")
print(f"Brier Score: {ens_brier:.4f} (Probability reliability - closer to 0 is better)")

print("\n--- 3. CONFUSION MATRIX BREAKDOWN ---")
print(f"True Positives (Deepfakes correctly caught) : {tp}")
print(f"True Negatives (Real images verified)       : {tn}")
print(f"False Positives (Real labeled as Fake)      : {fp}   <- Costly for user trust/reputation")
print(f"False Negatives (Deepfakes that slipped by) : {fn}   <- Critical security misses")

print("==========================================================================")

      🔬 COMPREHENSIVE ACADEMIC METRICS (VERSION 11 FINAL) 🔬

--- 1. INDIVIDUAL MODEL PERFORMANCE (AT OPTIMAL THRESHOLD) ---
Model Name                 | Thr  | Acc    | Prec   | Rec    | F1     | AUC   
--------------------------------------------------------------------------------
DINOv2-BASE @ 518px        | 0.48 | 0.9046 | 0.8773 | 0.9326 | 0.9041 | 0.9627
SigLIP-400M @ 224px        | 0.50 | 0.8962 | 0.8816 | 0.9067 | 0.8940 | 0.9582
CLIP ViT-L/14 @ 224px      | 0.50 | 0.8985 | 0.8853 | 0.9071 | 0.8961 | 0.9600
XGBoost (Offline Features) | 0.39 | 0.7419 | 0.6877 | 0.8514 | 0.7609 | 0.8319

--- 2. FINAL ENSEMBLE METRICS (CALIBRATED + HILL CLIMB) ---
Accuracy   : 0.9308 (Overall correctness across both classes)
Precision  : 0.9207 (When it says AI, it is actually AI this % of the time)
Recall     : 0.9374 (It successfully caught this % of all AI images)
F1 Score   : 0.9289 (The harmonic mean of Precision & Recall) @ Thr=0.51
ROC AUC    : 0.9783 (Overall class separation capability)
B

In [37]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, brier_score_loss, confusion_matrix,
                             average_precision_score, log_loss, matthews_corrcoef, fbeta_score)

print("==========================================================================")
print("      🔬 EXHAUSTIVE ACADEMIC METRICS (VERSION 11 FINAL) 🔬")
print("==========================================================================\n")

# --- 1. DATA RECOVERY ---
if 'y_all' not in globals():
    found_csv = False
    for search_dir in [Path('..'), Path('.')]:
        for p in search_dir.rglob('train.csv'):
            df = pd.read_csv(p)
            target_col = 'label' if 'label' in df.columns else 'target'
            y_all = df[target_col].values
            found_csv = True
            break
        if found_csv: break
    if not found_csv: raise ValueError("🚨 ERROR: Cannot find train.csv.")

# --- 2. RECOVER PREDICTIONS ---
CHKPT_DIR = Path('./chkpt_v11')
if 'oof_store' not in globals():
    oof_store = {}
    for m, p in [('siglip_ft', 'sig_oof.npy'), ('clip_ft', 'clip_oof.npy'), 
                 ('dino_ft', 'dino_oof.npy'), ('xgb', 'xgb_oof.npy')]:
        try: oof_store[m] = np.load(CHKPT_DIR / p)
        except: pass

v11_models = ['dino_ft', 'siglip_ft', 'clip_ft', 'xgb']
model_names = {
    'dino_ft': 'DINOv2-BASE @ 518px',
    'siglip_ft': 'SigLIP-400M @ 224px',
    'clip_ft': 'CLIP ViT-L/14 @ 224px',
    'xgb': 'XGBoost (Offline Features)'
}

# --- 3. FINAL ENSEMBLE METRICS ---
weights = {'dino_ft': 0.36, 'siglip_ft': 0.24, 'clip_ft': 0.20, 'xgb': 0.20} 
ens_proba = np.zeros_like(y_all, dtype=float)
for m in v11_models:
    if m in oof_store:
        prob = calibrated_oof[m] if 'calibrated_oof' in globals() and m in calibrated_oof else oof_store[m]
        ens_proba += prob * weights.get(m, 0)

# Find optimal threshold for ensemble
ens_best_f1, ens_best_t = 0, 0.50
for t in np.arange(0.35, 0.65, 0.01):
    f1 = f1_score(y_all, (ens_proba >= t).astype(int))
    if f1 > ens_best_f1: 
        ens_best_f1 = f1; ens_best_t = t

bin_ens_preds = (ens_proba >= ens_best_t).astype(int)

# --- CALCULATE ADVANCED METRICS ---
ens_acc = accuracy_score(y_all, bin_ens_preds)
ens_prec = precision_score(y_all, bin_ens_preds, zero_division=0)
ens_rec = recall_score(y_all, bin_ens_preds)
ens_auc = roc_auc_score(y_all, ens_proba)
ens_pr_auc = average_precision_score(y_all, ens_proba)
ens_brier = brier_score_loss(y_all, ens_proba)
ens_logloss = log_loss(y_all, ens_proba)
ens_mcc = matthews_corrcoef(y_all, bin_ens_preds)
ens_f2 = fbeta_score(y_all, bin_ens_preds, beta=2.0)

tn, fp, fn, tp = confusion_matrix(y_all, bin_ens_preds).ravel()
specificity = tn / (tn + fp)
far = fp / (fp + tn)  # False Acceptance Rate
frr = fn / (fn + tp)  # False Rejection Rate

print("--- ADVANCED CLASSIFICATION METRICS ---")
print(f"Accuracy         : {ens_acc:.4f}  (Overall Correctness)")
print(f"Precision        : {ens_prec:.4f}  (Trust: When accused of being fake, is it?)")
print(f"Recall (TPR)     : {ens_rec:.4f}  (Security: % of total fakes caught)")
print(f"Specificity (TNR): {specificity:.4f}  (Protection: % of real images verified)")
print(f"F1 Score         : {ens_best_f1:.4f}  (Harmonic mean of Prec & Rec) @ Thr={ens_best_t:.2f}")
print(f"F2 Score         : {ens_f2:.4f}  (Security-focused F-score weighting Recall higher)")
print(f"MCC              : {ens_mcc:.4f}  (Matthews Corr. Coef. - Best single metric for quality)")

print("\n--- ADVANCED PROBABILITY & RANKING METRICS ---")
print(f"ROC AUC          : {ens_auc:.4f}  (Separation between Real/Fake distributions)")
print(f"PR AUC (Avg Prec): {ens_pr_auc:.4f}  (Area under Precision-Recall curve)")
print(f"Brier Score      : {ens_brier:.4f}  (Mean squared error of probabilities)")
print(f"Log Loss         : {ens_logloss:.4f}  (Penalty for overconfident wrong predictions)")

print("\n--- FORENSIC SECURITY METRICS (CONFUSION MATRIX) ---")
print(f"True Positives (Fakes Caught)  : {tp}")
print(f"True Negatives (Reals Cleared) : {tn}")
print(f"False Positives (Reals Accused): {fp}  |  FAR (False Accept Rate): {far*100:.2f}%")
print(f"False Negatives (Fakes Missed) : {fn}  |  FRR (False Reject Rate): {frr*100:.2f}%")
print("==========================================================================")

      🔬 EXHAUSTIVE ACADEMIC METRICS (VERSION 11 FINAL) 🔬

--- ADVANCED CLASSIFICATION METRICS ---
Accuracy         : 0.9308  (Overall Correctness)
Precision        : 0.9207  (Trust: When accused of being fake, is it?)
Recall (TPR)     : 0.9374  (Security: % of total fakes caught)
Specificity (TNR): 0.9247  (Protection: % of real images verified)
F1 Score         : 0.9289  (Harmonic mean of Prec & Rec) @ Thr=0.51
F2 Score         : 0.9340  (Security-focused F-score weighting Recall higher)
MCC              : 0.8617  (Matthews Corr. Coef. - Best single metric for quality)

--- ADVANCED PROBABILITY & RANKING METRICS ---
ROC AUC          : 0.9783  (Separation between Real/Fake distributions)
PR AUC (Avg Prec): 0.9763  (Area under Precision-Recall curve)
Brier Score      : 0.0734  (Mean squared error of probabilities)
Log Loss         : 0.2725  (Penalty for overconfident wrong predictions)

--- FORENSIC SECURITY METRICS (CONFUSION MATRIX) ---
True Positives (Fakes Caught)  : 2170
True Negat

In [38]:
# ================================================================
# CELL 18: QUALITATIVE ERROR ANALYSIS (HARD EXAMPLE MINING)
# ================================================================
import matplotlib.pyplot as plt
import cv2

print("Mining hard examples for visual analysis...")

# ens_proba and y_all should be in memory from the previous metrics script
errors_df = pd.DataFrame({
    'path': train_paths,  # Assuming train_paths aligns with y_all and ens_proba
    'true_label': y_all,
    'pred_prob': ens_proba
})

# False Positives: True label is 0 (Real), but pred_prob is very high
fps = errors_df[(errors_df['true_label'] == 0) & (errors_df['pred_prob'] >= 0.5)].copy()
fps = fps.sort_values(by='pred_prob', ascending=False).head(5)

# False Negatives: True label is 1 (Fake), but pred_prob is very low
fns = errors_df[(errors_df['true_label'] == 1) & (errors_df['pred_prob'] < 0.5)].copy()
fns = fns.sort_values(by='pred_prob', ascending=True).head(5)

def plot_image_row(df_subset, title, filename):
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    fig.suptitle(title, fontsize=18, fontweight='bold')
    
    for ax, (_, row) in zip(axes, df_subset.iterrows()):
        img = cv2.imread(str(row['path']))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        ax.set_title(f"Pred: {row['pred_prob']:.3f}\nTrue: {row['true_label']}", color='red', fontweight='bold')
        ax.axis('off')
        
    plt.tight_layout()
    plt.savefig(IMG_DIR / filename, dpi=300)
    plt.show()

if len(fps) > 0:
    plot_image_row(fps, "Top 5 Worst False Positives (Reals predicted as Fake)", '11_Worst_False_Positives.png')
if len(fns) > 0:
    plot_image_row(fns, "Top 5 Worst False Negatives (Fakes predicted as Real)", '12_Worst_False_Negatives.png')

print("✅ Saved qualitative error grids to Images_v11 folder.")

Mining hard examples for visual analysis...
✅ Saved qualitative error grids to Images_v11 folder.


In [40]:
# ================================================================
# CELL 19 (ALTERNATIVE): XGBOOST "TIE-BREAKER" COMPLEMENTARITY
# ================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Analyzing XGBoost's unique contribution to the ensemble...")

# Calculate binary predictions for each model (using approx optimal thresholds)
dino_preds = (oof_store['dino_ft'] >= 0.48).astype(int)
siglip_preds = (oof_store['siglip_ft'] >= 0.50).astype(int)
clip_preds = (oof_store['clip_ft'] >= 0.50).astype(int)
xgb_preds = (oof_store['xgb'] >= 0.39).astype(int) # XGBoost optimal thr was 0.39

# 1. Find the "XGBoost Saves the Day" examples
# Cases where ALL Vision Transformers got it wrong, but XGBoost got it right
vit_all_wrong = (dino_preds != y_all) & (siglip_preds != y_all) & (clip_preds != y_all)
xgb_right = (xgb_preds == y_all)

xgb_saved_the_day = vit_all_wrong & xgb_right
num_saved = xgb_saved_the_day.sum()

print(f"\\n🔥 XGBoost successfully correctly classified {num_saved} images where ALL THREE Vision Transformers failed!")

# 2. Visualize Model Probability Distributions (Violin Plot)
# This shows WHY calibration was needed and how the models differ
print("Generating Prediction Distribution Violin Plots...")

pred_df = pd.DataFrame({
    'True Label': ['Fake' if y == 1 else 'Real' for y in y_all],
    'DINOv2': oof_store['dino_ft'],
    'SigLIP': oof_store['siglip_ft'],
    'CLIP': oof_store['clip_ft'],
    'XGBoost': oof_store['xgb']
})

# Melt the dataframe for Seaborn
melted_df = pd.melt(pred_df, id_vars=['True Label'], 
                    value_vars=['DINOv2', 'SigLIP', 'CLIP', 'XGBoost'],
                    var_name='Model', value_name='Predicted Probability')

plt.figure(figsize=(12, 6))
sns.violinplot(data=melted_df, x='Model', y='Predicted Probability', hue='True Label', 
               split=True, inner='quartile', palette={'Real': '#2ecc71', 'Fake': '#e74c3c'})

plt.title('Prediction Probability Distribution by Model\\n(Notice how XGBoost distribution differs entirely from the ViTs)', fontweight='bold', fontsize=14)
plt.axhline(0.5, color='black', linestyle='--', alpha=0.5)
plt.ylabel('Raw Predicted Probability')
plt.tight_layout()
try:
    plt.savefig(IMG_DIR / '13_Probability_Violins.png', dpi=300)
    print("✅ Saved Violin Plot to Images_v11/13_Probability_Violins.png")
except:
    plt.show()

# 3. XGBoost Unique Accuracy Contribution
# Create a Venn-diagram style bar chart
unique_correct = pd.DataFrame({
    'Model': ['DINOv2 Only', 'SigLIP Only', 'CLIP Only', 'XGBoost Only'],
    'Solely Correct Predictions': [
        ((dino_preds == y_all) & (siglip_preds != y_all) & (clip_preds != y_all) & (xgb_preds != y_all)).sum(),
        ((siglip_preds == y_all) & (dino_preds != y_all) & (clip_preds != y_all) & (xgb_preds != y_all)).sum(),
        ((clip_preds == y_all) & (dino_preds != y_all) & (siglip_preds != y_all) & (xgb_preds != y_all)).sum(),
        ((xgb_preds == y_all) & (dino_preds != y_all) & (siglip_preds != y_all) & (clip_preds != y_all)).sum()
    ]
})

plt.figure(figsize=(8, 5))
sns.barplot(data=unique_correct, x='Model', y='Solely Correct Predictions', palette='magma')
plt.title('Unique Contributions: Images ONLY this specific model got right', fontweight='bold')
plt.ylabel('Number of Images')
for index, row in unique_correct.iterrows():
    plt.text(index, row['Solely Correct Predictions'] + 2, str(row['Solely Correct Predictions']), color='black', ha="center", fontweight='bold')
plt.tight_layout()
try:
    plt.savefig(IMG_DIR / '14_Unique_Contributions.png', dpi=300)
    print("✅ Saved Unique Contributions to Images_v11/14_Unique_Contributions.png")
except:
    plt.show()

Analyzing XGBoost's unique contribution to the ensemble...
\n🔥 XGBoost successfully correctly classified 30 images where ALL THREE Vision Transformers failed!
Generating Prediction Distribution Violin Plots...
✅ Saved Violin Plot to Images_v11/13_Probability_Violins.png
✅ Saved Unique Contributions to Images_v11/14_Unique_Contributions.png


In [46]:
import zipfile
import os

# Files and directories to include
files = ['version_11_final (1).ipynb', 'submission.csv', 'analysis_v11.png']
dirs = ['models_v11', 'data/Images_v11', 'data/cache_v11', 'data/chkpt_v11', 'data/models_v11']
zip_name = 'version_11_complete_package.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
    # Add individual files
    for f in files:
        if os.path.exists(f):
            z.write(f)
            print(f"Added file: {f}")
            
    # Add entire directories
    for d in dirs:
        if os.path.exists(d):
            for root, _, filenames in os.walk(d):
                for name in filenames:
                    file_path = os.path.join(root, name)
                    z.write(file_path)
                    print(f"Added from dir: {name}")

print("\n--- ZIP COMPLETE ---")


Added file: submission.csv
Added file: analysis_v11.png
Added from dir: dino_fold_1.pt
Added from dir: clip_fold_3.pt
Added from dir: clip_fold_1.pt
Added from dir: clip_fold_4.pt
Added from dir: dino_fold_4.pt
Added from dir: sig_fold_2.pt
Added from dir: sig_fold_4.pt
Added from dir: dino_fold_0.pt
Added from dir: sig_fold_3.pt
Added from dir: clip_fold_0.pt
Added from dir: dino_fold_3.pt
Added from dir: dino_fold_2.pt
Added from dir: sig_fold_1.pt
Added from dir: clip_fold_2.pt
Added from dir: sig_fold_0.pt

--- ZIP COMPLETE ---
